# Lab 10: NLP Sentiment Analysis on Product Reviews

This notebook performs sentiment analysis, text preprocessing, word frequency analysis, word cloud generation, and POS tagging visualization on product reviews dataset.

**Note:** Since the product reviews CSV was not found in `datasets/`, we generate synthetic realistic product reviews data for this lab demonstration.

## Setup

Install required packages and download NLTK data.

In [ ]:
!pip install vaderSentiment wordcloud matplotlib nltk textblob --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import nltk
from wordcloud import WordCloud
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt', quiet=True)

print('Setup complete!')

## Load Data

Generate synthetic product reviews data (200 reviews).

In [ ]:
# Synthetic product reviews data
np.random.seed(42)
products = ['Headphones', 'Laptop', 'Phone', 'Tablet', 'Mouse', 'Keyboard', 'Monitor', 'Camera']

# Positive, neutral, negative templates
positive_reviews = [
    'Excellent product, great quality and value for money',
    'Love it! Works perfectly and looks amazing',
    'Highly recommend, fast delivery and good packaging',
    'Perfect! Exactly what I needed, very satisfied',
    'Amazing performance, battery life is fantastic',
    'Best purchase ever, exceeded my expectations'
]
negative_reviews = [
    'Poor quality, broke after one day',
    'Very disappointed, does not work as advertised',
    'Waste of money, returned immediately',
    'Bad product, slow and unreliable',
    'Terrible customer service, item defective',
    'Not worth the price, low quality materials'
]
neutral_reviews = [
    'Average product, meets basic requirements',
    'OK for the price, nothing special',
    'Works fine, standard quality',
    'As expected, no complaints',
    'Decent performance, good enough'
]

# Generate 200 reviews with sentiment distribution ~60% pos, 20% neg, 20% neut
reviews = []
for _ in range(120):
    prod = np.random.choice(products)
    rev = np.random.choice(positive_reviews)
    reviews.append(f"{rev}. The {prod} is really great.")
for _ in range(40):
    prod = np.random.choice(products)
    rev = np.random.choice(negative_reviews)
    reviews.append(f"{rev}. The {prod} is disappointing.")
for _ in range(40):
    prod = np.random.choice(products)
    rev = np.random.choice(neutral_reviews)
    reviews.append(f"{rev}. The {prod} performs adequately.")

# Shuffle
np.random.shuffle(reviews)

df = pd.DataFrame({'review_text': reviews})
print(f'Dataset shape: {df.shape}')
print('\nSample reviews:')
print(df.head())

## Task 1 – Sentiment Analysis + Bar Chart

Use VADER SentimentIntensityAnalyzer to classify reviews and plot distribution.

In [ ]:
# VADER sentiment analysis
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores = analyzer.polarity_scores(text)
    if scores['compound'] >= 0.05:
        return 'Positive'
    elif scores['compound'] <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment'] = df['review_text'].apply(get_sentiment)
sentiment_counts = df['sentiment'].value_counts().sort_index()

print('Sentiment distribution:')
print(sentiment_counts)

# Plot bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green', 'gray', 'red']  # Positive, Neutral, Negative
bars = ax.bar(sentiment_counts.index, sentiment_counts.values, color=colors)
ax.set_title('Sentiment Distribution of Reviews', fontsize=16, pad=20)
ax.set_xlabel('Sentiment', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)

# Add value labels on top of bars
for bar, count in zip(bars, sentiment_counts.values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{int(count)}', ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

## Task 2 – Text Preprocessing + Lemmatization

Clean text, tokenize, remove stopwords, lemmatize with POS tags.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# POS tag mapping for lemmatization
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'  # adjective
    elif tag.startswith('V'):
        return 'v'  # verb
    elif tag.startswith('N'):
        return 'n'  # noun
    elif tag.startswith('R'):
        return 'r'  # adverb
    else:
        return 'n'  # default noun

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation and digits
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    words = word_tokenize(text)
    # Remove stopwords
    words = [w for w in words if w not in stop_words and len(w) > 2]
    # POS tag and lemmatize
    tagged = pos_tag(words)
    lemmatized = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]
    return lemmatized

# Apply preprocessing
df['processed_tokens'] = df['review_text'].apply(preprocess_text)

# Full corpus for frequency analysis
full_corpus = [token for tokens in df['processed_tokens'] for token in tokens]

print('Sample processed tokens:', df['processed_tokens'].iloc[0])
print(f'Full corpus size: {len(full_corpus)} tokens')

## Task 3 – Top 15 Most Frequent Words Bar Chart

Horizontal bar chart of top 15 words.

In [ ]:
# Word frequency
word_freq = Counter(full_corpus).most_common(15)
words, frequencies = zip(*word_freq)

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(words)), frequencies, color='steelblue')
ax.set_title('Top 15 Most Frequent Words', fontsize=16, pad=20)
ax.set_xlabel('Frequency', fontsize=12)
ax.set_ylabel('Words', fontsize=12)
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)

# Add frequency labels
for i, (bar, freq) in enumerate(zip(bars, frequencies)):
    ax.text(freq + 1, bar.get_y() + bar.get_height()/2, str(freq),
            va='center', fontsize=11)

plt.tight_layout()
plt.show()

## Task 4 – Word Cloud

Word cloud from lemmatized corpus.

In [ ]:
# Generate word cloud
corpus_text = ' '.join(full_corpus)
wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=200).generate(corpus_text)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wordcloud, interpolation='bilinear')
ax.set_title('Word Cloud of Lemmatized Reviews', fontsize=16, pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

## Task 5 – POS Frequency Bar Chart

Count Nouns (NN*), Verbs (VB*), Adjectives (JJ*) and plot.

In [ ]:
# POS tagging on full corpus
pos_tags = pos_tag(full_corpus)

# Count POS categories
nouns = sum(1 for word, tag in pos_tags if tag.startswith(('NN', 'NNS', 'NNP', 'NNPS')))
verbs = sum(1 for word, tag in pos_tags if tag.startswith('VB'))
adjs = sum(1 for word, tag in pos_tags if tag.startswith('JJ'))

pos_counts = {'Nouns': nouns, 'Verbs': verbs, 'Adjectives': adjs}

print('POS frequencies:', pos_counts)

# Plot bar chart
fig, ax = plt.subplots(figsize=(10, 6))
categories = list(pos_counts.keys())
counts = list(pos_counts.values())
colors = ['orange', 'skyblue', 'lightgreen']
bars = ax.bar(categories, counts, color=colors)
ax.set_title('POS Tag Frequency in Reviews', fontsize=16, pad=20)
ax.set_xlabel('Part of Speech', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)

# Add value labels
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{count}', ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

print('\nLab 10 complete! All tasks implemented successfully.')